In [ ]:
# WORKING VERSION, except for matrices of type LONG

import socket
import struct
import time
import numpy as np

JIT_MATRIX_PACKET_ID = 0x4A4D5458  # 'JMTX'
MAX_DIMS = 32
MATRIX_HEADER_SIZE = 288

TYPE_CHAR    = 0
TYPE_LONG    = 1
TYPE_FLOAT32 = 2
TYPE_FLOAT64 = 3

def swap32(x: int) -> bytes:
    # uint32 big-endian
    return struct.pack(">I", x)
    
# ============================================================
# Build TCP packet for a given matrix
# ============================================================
def build_packet(matrix: np.ndarray):
    if matrix.ndim < 2:
        raise ValueError("Matrix must have at least 2 dimensions (P, rows, cols)")

    planecount = matrix.shape[0]
    if planecount > 32:
        raise ValueError("Planecount cannot be higher than 32")
   
    raw_dims = list(matrix.shape[1:])  # [rows, cols, ...]
    dimcount = len(raw_dims)

    if dimcount > MAX_DIMS:
        raise ValueError(f"Too many dimensions for Jitter (max {MAX_DIMS})")

    # NumPy: (P, rows, cols)
    # Max:   dim[0] = cols, dim[1] = rows
    if dimcount >= 2:
        H = raw_dims[0]
        W = raw_dims[1]
        dims = [W, H] + raw_dims[2:]
    else:
        dims = raw_dims[:]

    mat = matrix  # (P, rows, cols, ...)

    # ---------- endian conversion BEFORE interleaving ----------
    if mat.dtype == np.uint8:
        jittype = TYPE_CHAR
        bytes_per_plane = 1
        mat_be = mat

    elif mat.dtype == np.int32:
        jittype = TYPE_LONG
        bytes_per_plane = 8
        # IMPORTANT: no endian conversion for LONG
        mat_be = mat.astype('>i4', copy=True)

    elif mat.dtype == np.float32:
        jittype = TYPE_FLOAT32
        bytes_per_plane = 4
        mat_be = mat.astype('>f4', copy=True)

    elif mat.dtype == np.float64:
        jittype = TYPE_FLOAT64
        bytes_per_plane = 8
        mat_be = mat.astype('>f8', copy=True)

    else:
        raise ValueError(f"dtype unsupported : {mat.dtype}")

    # ---------- interleave planes: (P, D1, D2, ...) → (D1, D2, ..., P) ----------
    axes = list(range(1, mat_be.ndim)) + [0]
    mat_be = np.transpose(mat_be, axes)
    send_arr = np.ascontiguousarray(mat_be)

    # ---------- dimstride ----------
    dim = dims + [0] * (MAX_DIMS - dimcount)
    dimstride = [0] * MAX_DIMS
    dimstride[0] = planecount * bytes_per_plane
    for i in range(1, dimcount):
        dimstride[i] = dimstride[i - 1] * dim[i - 1]

    datasize = send_arr.nbytes

    # ---------- matrix header ----------
    mh = bytearray()
    mh += struct.pack("<I", JIT_MATRIX_PACKET_ID)
    mh += swap32(MATRIX_HEADER_SIZE)
    mh += swap32(planecount)
    mh += swap32(jittype)
    mh += swap32(dimcount)
    for v in dim:
        mh += swap32(v)
    for v in dimstride:
        mh += swap32(v)
    mh += swap32(datasize)
    mh += struct.pack("<d", time.time())

    # ---------- chunk header ----------
    ch = bytearray()
    ch += swap32(JIT_MATRIX_PACKET_ID)
    ch += swap32(len(mh) + datasize)

    return bytes(ch), bytes(mh), send_arr.tobytes(order="C")


# ============================================================
# TCP client
# ============================================================

class JitterTCPClient:
    def __init__(self):
        self.sock = None
        self.running = False
        self.thread = None
        self.fps = 30
        self.width = 20
        self.height = 20
        self.planes = 3
        self.dtype = np.float32
        self.matrix_generator = None
        
    def connect(self, host="127.0.0.1", port=7474):
        """Établit la connexion TCP avec Max"""
        self.sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.sock.connect((host, port))
        print(f"Connecté à {host}:{port}")

    def sendmatrix(self, matrix: np.ndarray, plane_axis=None):
        """Envoie une matrice numpy à Max"""
        if self.sock is None:
            raise RuntimeError("TCP non connecté")
        chunk_header, matrix_header, matrix_data = build_packet(matrix)
        self.sock.sendall(chunk_header)
        self.sock.sendall(matrix_header)
        self.sock.sendall(matrix_data)

    def disconnect(self):
        """Ferme la connexion TCP"""
        if self.sock is not None:
            self.sock.close()
            self.sock = None
            print("Connexion fermée")


# ============================================================
# Main – boucle d’envoi continu avec framerate
# ============================================================

if __name__ == "__main__":
    client = JitterTCPClient()
    client.connect(host="127.0.0.1", port=7474)

    # Test matrices for various types
    # ===============================
    ####### Single line CHAR matrix
    #mat = np.arange(4, dtype=np.uint8).reshape((1,4))
    ####### Single plane CHAR matrix
    #mat = np.arange(6, dtype=np.uint8).reshape((1,3,2))  
    ####### Two planes CHAR matrix
    mat = np.arange(24, dtype=np.uint8).reshape((2,3,4))
    ####### Single plane FLOAT32 matrix
    #mat = np.arange(12, dtype=np.float32).reshape((1,3,4))
    ####### Two planes FLOAT32 matrix
    #mat = np.arange(18, dtype=np.float32).reshape((2,3,3))
    ####### Three planes FLOAT32 240 high, 320 wide
    #mat = np.random.rand(3, 240, 320).astype(np.float32) 
    ####### Single plane FLOAT64 matrix
    #mat = np.arange(12, dtype=np.float64).reshape((1,3,4))
    ####### Two planes FLOAT64 matrix
    #mat = np.arange(24, dtype=np.float64).reshape((2,3,4))
    ####### Single plane LONG matrix
    #mat = np.arange(12, dtype=np.int32).reshape((1,3,4))
    ####### Two planes LONG matrix
    #mat = np.arange(24, dtype=np.int32).reshape((2,3,4)) 

    # planes are axis 0
    #mat = np.arange(120, dtype=np.uint8).reshape((2,3,4,5))
    
    client.sendmatrix(mat)

    print(mat)
    print(mat.shape)

    # leave some time to check if connected attribute correctly switched to ON in Max
    time.sleep(3)
    
    client.disconnect()
